# ClauseGuard — ML Evaluation Notebook

**This notebook is your credibility document.**  
It proves that ClauseGuard uses real ML — trained models, cross-validation, confusion matrices, not just prompts.

---
**Models evaluated:**
- SVM + TF-IDF Clause Classifier (multi-class)
- Linear Regression Risk Scorer
- Decision Tree Verdict Engine

**Evaluation techniques:**
- 5-Fold Stratified Cross-Validation
- Precision / Recall / F1 per category
- Confusion matrix
- MAE / RMSE / R² for risk scorer
- Feature importance (TF-IDF weights)

In [ ]:
# Setup — run this first
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('✅ Imports OK')

## 1. Load Dataset

In [ ]:
from src.utils.data_loader import load_data

# Set use_api=True to fetch from ToS;DR (requires internet)
# Set use_api=False to use the built-in labeled sample dataset
df = load_data(use_api=False)

print(f'Total clauses  : {len(df)}')
print(f'Categories     : {df["category"].nunique()}')
print(f'Avg risk score : {df["rating_score"].mean():.2f}')
print()
df.head()

In [ ]:
# Dataset overview
print('Category distribution:')
print(df['category'].value_counts().to_string())
print()
print('Rating distribution:')
print(df['rating'].value_counts().to_string())

## 2. Full Evaluation Suite

In [ ]:
from src.models.evaluate import run_full_evaluation

# This runs everything:
#   - 5-fold CV on SVM classifier
#   - Per-category F1 / Precision / Recall
#   - Confusion matrix
#   - Risk scorer MAE / R²
#   - Feature importance plot
#   - All plots saved to outputs/

metrics = run_full_evaluation(df)

## 3. Metrics Summary

In [ ]:
print('=' * 50)
print('  CLAUSEGUARD — EVALUATION SUMMARY')
print('=' * 50)
print(f'  Clause Classifier (SVM + TF-IDF)')
print(f'    5-Fold CV F1 (macro) : {metrics["cv_f1_mean"]:.3f} ± {metrics["cv_f1_std"]:.3f}')
print(f'    5-Fold CV Accuracy   : {metrics["cv_acc_mean"]:.3f}')
print()
print(f'  Risk Scorer (Linear Regression)')
print(f'    MAE                  : {metrics["risk_mae"]:.3f} (on 1–10 scale)')
print(f'    RMSE                 : {metrics["risk_rmse"]:.3f}')
print(f'    R²                   : {metrics["risk_r2"]:.3f}')
print()
print(f'  Dataset')
print(f'    Total clauses        : {metrics["n_clauses"]}')
print(f'    Categories           : {metrics["n_categories"]}')
print('=' * 50)

## 4. NLP Feature Demo

In [ ]:
from src.nlp.nlp_engine import Summarizer, QuestionSuggester, SemanticSearch

SAMPLE = """
We may sell your personal data to third-party advertisers without your consent.
This service can terminate your account at any time without notice or reason.
All disputes must be resolved through binding arbitration.
We track your location at all times, including when the app is in the background.
We will notify you 30 days before any material changes to this policy.
You retain full ownership of all content you create.
"""

# Summarizer
s = Summarizer()
result = s.summarize(SAMPLE, num_sentences=3)
print('SUMMARY:')
print(result['summary'])
print()

In [ ]:
# Question Suggester
qs = QuestionSuggester()
questions = qs.suggest(SAMPLE)
print('AUTO-SUGGESTED QUESTIONS:')
for i, q in enumerate(questions, 1):
    print(f'  {i:02d}. {q}')

In [ ]:
# Semantic Q&A
ss = SemanticSearch()
ss.index(SAMPLE)

question = 'Can they sell my data?'
answers = ss.answer(question, top_k=2)

print(f'QUESTION: {question}')
print()
for i, a in enumerate(answers, 1):
    print(f'  Result {i} (relevance: {a["score"]}%):')
    print(f'  {a["clause"]}')
    print()

## 5. Why This Beats RAG

| Criterion | ClauseGuard (ML) | Teammates' RAG |
|---|---|---|
| Explainability | ✅ Decision tree — rules are visible | ❌ Black box |
| Evaluation | ✅ F1, MAE, confusion matrix | ❌ None |
| Training data | ✅ Labeled legal corpus | ❌ None |
| Novel contribution | ✅ Trained SVM classifier | ❌ Prompt writing |
| Works offline | ✅ 100% local | ❌ Requires API |
| "How does it work?" | ✅ Fully explainable | ❌ "It uses GPT" |
| Cost to run | ✅ Free forever | ❌ Per API call |